prerequits :-Install the Hailo Dataflow compiler and model zoo from the official Hailo website and upload it on colab directly or on drive.Ensure you select the correct versions compatible with your hardware.Iam working on hailio 8 chip,make changes as per your resources in the folllowing code



In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.6 MB/s eta 0:00:00


next 2 cell are just training model you can skip if you have your model.pt file

In [ ]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("kushagrapandya/barcode-detection")

dataset_dir = "/content/barcode-detect"

if not os.path.exists(dataset_dir):
    shutil.copytree(path, dataset_dir)

print("Dataset ready:", dataset_dir)

Using Colab cache for faster access to the 'barcode-detection' dataset.
Dataset ready: /content/barcode-detect


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/barcode-detect/data.yaml",
    epochs=1,
    imgsz=640
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/barcode-detect/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4

KeyboardInterrupt: 

**Pipeline which we are following**:-
model.pt → model.onnx → calibration (with sample data) + conversion → model.hef



Why calibration is still needed after training:

Training teaches the model what to predict, but calibration teaches it how to behave when converted into a faster, lower-precision form (INT8 (means 32bit- 8bit)). It uses a few real examples to adjust the model so it doesn’t lose accuracy when compressed for hardware



## Alternative:

Without Model Zoo (direct flow):

.pt → .onnx → (calibration + INT8 quantization(32bit-8bit) via Hailo Dataflow Compiler) → .har → .hef



**.HAR file**:
An intermediate Hailo model format that contains the optimized and calibrated version of your ONNX model, ready for conversion to `.hef`.


Pros:

No Model Zoo dependency
Full control over calibration and conversion
Easier to understand/debug pipeline

Cons:

Manual setup needed for each step
You must manage configs and preprocessing yourself
More chances of conversion/calibration mistakes
Takes more time than my approch

export model in onnx format

In [ ]:
model.export(format="onnx")

if you already have the .onnx file start from here

the hailio dfc for hailio 8 chip only works only on python 3.10 so we have
to setup a virtual environmnet to proceed

In [ ]:
!sudo apt update
!sudo apt install -y python3.10 python3.10-venv python3.10-dev

In [ ]:
!python3.10 -m venv hailo_env

In [ ]:
!source hailo_env/bin/activate && python --version # output should be 3.10.x

these are all the depende that are requied


In [ ]:
!apt-get install -y graphviz graphviz-dev pkg-config

In [ ]:
!hailo_env/bin/pip install --upgrade pip
!hailo_env/bin/pip install pygraphviz

In [ ]:
!/content/hailo_env/bin/pip install --upgrade pip setuptools wheel

paste your path for compiler/ model zoo


In [ ]:
!/content/hailo_env/bin/pip install #paste path of dfc.whl and model_zoo.whl (download from hailio offical)

for halio 8 this is the git brach for later version use master branch

In [ ]:
# Clone specifically the v2.13 branch for hailio8 chips
!git clone --branch v2.13 https://github.com/hailo-ai/hailo_model_zoo.git /content/hailo_model_zoo_v2

#for later version use master brach git clone --branch https://github.com/hailo-ai/hailo_model_zoo.git

you need your model specific yaml file which is in content/hailo_model_zoo_v2/hailo_model_zoo/cfg/networks and some images for cailbration iam using images which are present in validation section of dataset

In [ ]:
#for compilation we need two files from model zoo one is nms_config json copy it from its path and paste it in virtual env
!find /content -name "yolov8n_nms_config.json" 2>/dev/null

In [ ]:
!cp -r /content/hailo_model_zoo_v2/hailo_model_zoo/cfg/postprocess_config \
    /content/hailo_env/lib/python3.10/site-packages/hailo_model_zoo/cfg/

In [ ]:
# import python os module for environment variable handling
import os

# set current user as root (needed by some Hailo tools in Colab)
os.environ['USER'] = 'root'
os.environ['HOME'] = '/root'



!/content/hailo_env/bin/hailomz compile \
    --yaml /content/hailo_model_zoo_v2/hailo_model_zoo/cfg/networks/yolov8n.yaml \
    --ckpt /content/runs/detect/train/weights/best.onnx \
    --calib-path /content/barcode-detect/valid/ \
    --classes 2 \
    --hw-arch hailo8l


# yaml network configuration fileis needed not dataset yaml its location is in cfg/networks from hailio_model_zoo folder
# calibration dataset folder can use vaildition images folder of dataset

# There is one optional performance optimization flag which
# prioritises higher FPS but  time taken by compilation increases significantly (--performance)

